# One-off: deduplicar bronce (1 fila por clave)

Colapsa **una** tabla de `bronze` a su estado actual: deja **una fila por
clave** (la de la captura mas reciente, con `_ingest_date` como desempate) y
**conserva sin tocar** las filas sin clave usable, que no se pueden agrupar y
no deben perderse.

- **Coste:** ~0,05-0,13 EUR por run. Lo domina el arranque del cluster
  `Standard_D4ds_v4` single node (~6-7 min a ~0,75 EUR/h); las cuatro tareas
  tardan segundos. No usa el SQL warehouse.
- **Seguridad:** `dry_run=true` cuenta antes y despues y termina sin escribir;
  la escritura es un unico `INSERT OVERWRITE` atomico y la celda de historial
  imprime la sentencia `RESTORE` para volver a la version anterior.
- **Cuando:** solo se lanza desde el job `dedupe_oneoff` (T-04 en dev, T-06 en
  prod). No forma parte del job diario ni semanal.

No toca el landing ni el checkpoint: solo reescribe la tabla bronce indicada
por `catalog` + `source`.


In [ ]:
# Databricks notebook source
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

import json
import os

import Upsert_Job_Offers as upsert

# ---------------------------------------------------------------------------
# Externalized configuration (Databricks widgets / environment variables).
# ---------------------------------------------------------------------------
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("source", "")
dbutils.widgets.dropdown("dry_run", "false", ["false", "true"])


def _cfg(name: str, default: str = "") -> str:
    """Resolve a parameter: widget -> environment variable -> default."""
    value = dbutils.widgets.get(name)
    return value if value else os.environ.get(name.upper(), default)


CATALOG = _cfg("catalog")
SOURCE = _cfg("source")
DRY_RUN = _cfg("dry_run", "false")

if not CATALOG:
    raise ValueError(
        "Falta el parametro 'catalog': indica el catalogo que contiene bronze.")
if not SOURCE:
    raise ValueError(
        "Falta el parametro 'source': indica la fuente a deduplicar.")
if DRY_RUN not in {"true", "false"}:
    raise ValueError(
        f"Valor no valido para 'dry_run': {DRY_RUN!r}. Usa 'false' o 'true'.")

# Fail fast on an unknown source before running any query.
upsert.get_source_spec(SOURCE)

# The ranking compares capture instants: pin the session zone to UTC, as the
# ingest does, so the order does not depend on the cluster default.
spark.conf.set("spark.sql.session.timeZone", "UTC")

TABLE = f"{CATALOG}.bronze.{SOURCE}"


In [ ]:
KEY_COLUMNS = upsert.get_source_spec(SOURCE).keys
KEY_LIST = ", ".join(f"`{name}`" for name in KEY_COLUMNS)
VALID_KEY = upsert.valid_key_sql(KEY_COLUMNS)

# One query for the three before counts: total rows, distinct usable keys and
# rows with a usable key. Rows without a usable key are preserved as they are.
COUNT_SQL = f"""
SELECT
  count(*) AS rows,
  count(DISTINCT CASE WHEN {VALID_KEY} THEN struct({KEY_LIST}) END)
    AS distinct_keys,
  count(CASE WHEN {VALID_KEY} THEN 1 END) AS valid_rows
FROM {TABLE}
"""

before = spark.sql(COUNT_SQL).first()
print(f"Antes de la deduplicacion en {TABLE}:")
print(f"  filas:                  {before['rows']}")
print(f"  claves distintas:       {before['distinct_keys']}")
print(f"  filas con clave valida: {before['valid_rows']}")
print(f"  filas sin clave usable: {before['rows'] - before['valid_rows']}")


In [ ]:
# Time travel: remember the current version so the run can be undone.
RESTORE_SQL = None
current_version = None
try:
    history = spark.sql(f"DESCRIBE HISTORY {TABLE} LIMIT 1").collect()
    current_version = history[0]["version"]
    RESTORE_SQL = f"RESTORE TABLE {TABLE} TO VERSION AS OF {current_version}"
    print(f"Version actual de {TABLE}: {current_version}")
    print(f"Rollback: {RESTORE_SQL}")
except Exception as exc:  # noqa: BLE001 - a table without history is valid
    print(
        f"{TABLE} no tiene historial de versiones: "
        f"no hay version de rollback ({exc}).")


In [ ]:
if DRY_RUN == "true":
    print("DRY RUN: no se escribe nada")
    print(f"{TABLE} queda como esta; relanza con dry_run=false para aplicar.")
    summary = {
        "mode": "dry_run",
        "source": SOURCE,
        "table": TABLE,
        "rows": int(before["rows"]),
        "distinct_keys": int(before["distinct_keys"]),
        "valid_rows": int(before["valid_rows"]),
        "version": (
            int(current_version) if current_version is not None else None),
        "dry_run": True,
    }
    dbutils.notebook.exit(
        json.dumps(summary, ensure_ascii=False, sort_keys=True))


In [ ]:
# One atomic rewrite: valid-key rows keep the latest capture, rows without a
# usable key are copied untouched, and the ranking column stays in the subquery.
spark.sql(
    upsert.dedupe_overwrite_sql(
        TABLE, SOURCE, spark.table(TABLE).columns))
print(f"Deduplicacion aplicada en {TABLE}.")


In [ ]:
after = spark.sql(COUNT_SQL).first()
rows_without_key = before["rows"] - before["valid_rows"]
expected_rows = after["distinct_keys"] + rows_without_key

print(f"Despues de la deduplicacion en {TABLE}:")
print(f"  filas:                  {after['rows']}")
print(f"  claves distintas:       {after['distinct_keys']}")
print(f"  filas sin clave usable: {rows_without_key}")

if after["rows"] != expected_rows:
    raise ValueError(
        "La deduplicacion no ha dejado una fila por clave en "
        f"{TABLE}: filas={after['rows']}, claves={after['distinct_keys']}, "
        f"filas sin clave usable={rows_without_key} "
        f"(esperado={expected_rows}).")

print("Comprobacion correcta: una fila por clave y filas sin clave conservadas.")

if RESTORE_SQL:
    print(f"Rollback disponible: {RESTORE_SQL}")
else:
    print("Rollback: la tabla no tenia historial de versiones.")

version_before = (
    int(current_version) if current_version is not None else None)
rows_after = int(after["rows"])
distinct_keys_after = int(after["distinct_keys"])
valid_rows_after = int(after["valid_rows"])
invalid_rows_after = rows_after - valid_rows_after
duplicates_after = valid_rows_after - distinct_keys_after

summary = {
    "mode": "apply",
    "source": SOURCE,
    "table": TABLE,
    "dry_run": False,
    "version_before": version_before,
    "rows_before": int(before["rows"]),
    "distinct_keys_before": int(before["distinct_keys"]),
    "valid_rows_before": int(before["valid_rows"]),
    "rows_after": rows_after,
    "distinct_keys_after": distinct_keys_after,
    "invalid_rows_after": invalid_rows_after,
    "duplicates_after": duplicates_after,
    "restore": RESTORE_SQL,
}
dbutils.notebook.exit(
    json.dumps(summary, ensure_ascii=False, sort_keys=True))
